# Exercice 30b (cloud) : aligner un petit assistant sur des préférences, avec DPO

`HuggingFaceTB/SmolLM2-360M-Instruct` (360 millions de paramètres, Apache-2.0) a déjà subi un
réglage par instructions : il répond. On lui apprend maintenant à **préférer** certaines réponses,
à partir de paires « réponse choisie / réponse rejetée » d'UltraFeedback (réponses de plusieurs
modèles notées par GPT-4, version binarisée de Hugging Face, licence MIT).

Le RLHF classique entraînerait d'abord un modèle de récompense, puis la politique par PPO, avec
quatre réseaux en mémoire. **DPO** (Rafailov et al., 2023) n'en garde que deux, la politique
$\pi_\theta$ et la référence figée $\pi_{\text{ref}}$, et ici même un seul jeu de poids : on
entraîne un **LoRA**, et la référence est le même modèle avec l'adaptateur désactivé.

**Perte optimisée** (exercice 30c pour la dérivation, sur un exemple jouet) :
$$\mathcal{L}_{\text{DPO}} = -\,\mathbb{E}\Big[\log \sigma\Big(\beta \log \tfrac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)}
- \beta \log \tfrac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\Big)\Big], \quad \beta = 0{,}1.$$
La quantité $\beta \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)}$ est la **récompense
implicite** de la réponse $y$ ; TRL l'affiche sous le nom `rewards/chosen` et `rewards/rejected`.

Plan : (1) les données et leur biais de longueur ; (2) la perte DPO calculée **à la main** sur une
paire ; (3) le modèle avant DPO ; (4) l'entraînement avec TRL ; (5) après : récompenses
implicites, précision sur des paires de test, longueur des réponses.

Avant de lancer : *Exécution › Modifier le type d'exécution › GPU T4*, puis *Exécution › Tout
exécuter*. Seed 42, **float16**. Aucun token Hugging Face n'est nécessaire.

In [ ]:
# Cellule 1 : vérifier le GPU
!nvidia-smi
import os
# Kaggle (« GPU T4 x2 ») : TRL refuse de tourner avec deux GPU visibles sans lanceur distribué.
# On n'en garde qu'un, avant d'importer torch. Sur Colab, cette ligne ne fait rien.
if os.path.exists("/kaggle/working") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import torch
TEST_LOCAL = os.environ.get("IA_30B_TEST") == "1"   # vérification du code en miniature, hors GPU (README)
if not TEST_LOCAL and not torch.cuda.is_available():
    raise SystemExit("Pas de GPU : Exécution › Modifier le type d'exécution › GPU T4, puis relancer.")
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# Cellule 2 : dépendances épinglées (torch n'est PAS épinglé : on garde celui de l'image Colab)
!pip install -q transformers==5.19.0 trl==1.15.0 peft==0.21.2 accelerate==1.15.0 datasets==5.1.0

In [ ]:
# Cellule 3 : configuration
import gc, json, random, time
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# COURT = True : 30 pas, pour vérifier que tout tourne
COURT = False
config = dict(
    modele="HuggingFaceTB/SmolLM2-360M-Instruct",          # Apache-2.0, non restreint
    donnees="HuggingFaceH4/ultrafeedback_binarized",       # MIT ; découpes train_prefs / test_prefs
    n_train=2000, n_test=200,
    max_car_prompt=600, max_car_reponse=1500,              # filtre de longueur (caractères)
    max_length=640,                                        # prompt + réponse, en tokens
    beta=0.1,                                              # force du rappel vers la référence
    lr=5e-5, rang=16, alpha=32,
    n_pas=30 if COURT else 250,
    lot=4, accumulation=2,                                 # lot effectif : 8 paires
    n_generations=40, nouveaux_tokens=256,
)
dispositif = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
if TEST_LOCAL:   # miniature : quelques pas, float32 (ne vérifie que le code)
    config.update(n_train=32, n_test=16, n_pas=4, lot=2, n_generations=4, nouveaux_tokens=40, max_length=384)
FP16 = dispositif == "cuda"
print(config, dispositif, "float16" if FP16 else "float32")

In [ ]:
# Cellule 4 : les données de préférences
from datasets import load_dataset

def preparer(decoupe, n):
    ds = load_dataset(config["donnees"], split=decoupe)
    ds = ds.filter(lambda e: len(e["prompt"]) <= config["max_car_prompt"]
                   and len(e["chosen"][-1]["content"]) <= config["max_car_reponse"]
                   and len(e["rejected"][-1]["content"]) <= config["max_car_reponse"]
                   and e["score_chosen"] > e["score_rejected"])          # pas d'égalités
    ds = ds.shuffle(seed=SEED).select(range(n))
    # Format conversationnel de TRL : prompt = messages avant la réponse ; chosen / rejected = la réponse
    return ds.map(lambda e: {"prompt": e["chosen"][:-1], "chosen": e["chosen"][-1:],
                             "rejected": e["rejected"][-1:]},
                  remove_columns=[c for c in ds.column_names if c not in ("prompt", "chosen", "rejected")])

train = preparer("train_prefs", config["n_train"])
test = preparer("test_prefs", config["n_test"])
ex = train[0]
print("PROMPT :", ex["prompt"][-1]["content"][:300])
print("\nCHOISIE :", ex["chosen"][0]["content"][:300])
print("\nREJETÉE :", ex["rejected"][0]["content"][:300])

lg_c = np.array([len(e["chosen"][0]["content"]) for e in train])
lg_r = np.array([len(e["rejected"][0]["content"]) for e in train])
print(f"\nLongueur moyenne (caractères) : choisies {lg_c.mean():.0f}, rejetées {lg_r.mean():.0f} ; "
      f"la choisie est la plus longue dans {np.mean(lg_c > lg_r):.0%} des paires")

In [ ]:
# Cellule 5 : charger le modèle, et la perte DPO calculée à la main sur une paire
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(config["modele"])
dtype = torch.float16 if FP16 else torch.float32
modele = AutoModelForCausalLM.from_pretrained(config["modele"], dtype=dtype).to(dispositif)
n_params = sum(p.numel() for p in modele.parameters())
print(f"{n_params / 1e6:.0f} M de paramètres")

@torch.no_grad()
def logp_reponse(m, prompt, reponse):
    # log π(y | x) = somme des log-probabilités des tokens de la réponse, sachant le prompt
    texte_prompt = tok.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
    texte_complet = tok.apply_chat_template(prompt + reponse, tokenize=False)
    ids_p = tok(texte_prompt, add_special_tokens=False)["input_ids"]
    ids = tok(texte_complet, add_special_tokens=False, return_tensors="pt")["input_ids"][:, :config["max_length"]].to(m.device)
    logits = m(ids).logits[:, :-1].float()
    lp = torch.log_softmax(logits, -1).gather(-1, ids[:, 1:, None]).squeeze(-1)
    return lp[0, len(ids_p) - 1:].sum().item()       # seuls les tokens de la réponse comptent

def perte_dpo_main(lp_w, lp_l, lp_w_ref, lp_l_ref, beta=config["beta"]):
    r_w, r_l = beta * (lp_w - lp_w_ref), beta * (lp_l - lp_l_ref)     # récompenses implicites
    return -F.logsigmoid(torch.tensor(r_w - r_l)).item(), r_w, r_l

e = test[0]
lp_w = logp_reponse(modele, e["prompt"], e["chosen"])
lp_l = logp_reponse(modele, e["prompt"], e["rejected"])
print(f"log π(y_w|x) = {lp_w:.1f} ; log π(y_l|x) = {lp_l:.1f}")
# Au départ, π_θ = π_ref : les deux log-ratios sont nuls et la perte vaut log 2
perte, r_w, r_l = perte_dpo_main(lp_w, lp_l, lp_w, lp_l)
print(f"Au départ : récompenses implicites {r_w:.3f} et {r_l:.3f}, perte = {perte:.4f} (log 2 = {np.log(2):.4f})")
# Si l'entraînement montait log π(y_w|x) de 5 et descendait log π(y_l|x) de 5 :
perte, r_w, r_l = perte_dpo_main(lp_w + 5, lp_l - 5, lp_w, lp_l)
print(f"Exemple : +5 / −5 nats → récompenses {r_w:+.1f} et {r_l:+.1f}, marge {r_w - r_l:.1f}, perte = {perte:.4f}")

In [ ]:
# Cellule 6 : AVANT DPO, préférences du modèle et longueur de ses réponses
@torch.no_grad()
def generer(m, prompts):
    m.eval()
    sorties = []
    for p in prompts:
        texte = tok.apply_chat_template(p, tokenize=False, add_generation_prompt=True)
        entree = tok(texte, return_tensors="pt", add_special_tokens=False).to(m.device)
        gen = m.generate(**entree, max_new_tokens=config["nouveaux_tokens"], do_sample=False,
                         pad_token_id=tok.pad_token_id or tok.eos_token_id)
        nouveaux = gen[0, entree["input_ids"].shape[1]:]
        sorties.append(dict(texte=tok.decode(nouveaux, skip_special_tokens=True), n_tokens=int(len(nouveaux))))
    return sorties

def preferences_modele(m, donnees):
    # Le modèle donne-t-il plus de probabilité à la réponse choisie ? (somme, puis moyenne par token)
    somme, moyenne = [], []
    for e in donnees:
        lw, ll = logp_reponse(m, e["prompt"], e["chosen"]), logp_reponse(m, e["prompt"], e["rejected"])
        nw = len(tok(e["chosen"][0]["content"])["input_ids"]); nl = len(tok(e["rejected"][0]["content"])["input_ids"])
        somme.append(lw > ll); moyenne.append(lw / nw > ll / nl)
    return float(np.mean(somme)), float(np.mean(moyenne))

t0 = time.perf_counter()
pref_avant = preferences_modele(modele, test)
prompts_gen = [e["prompt"] for e in test.select(range(config["n_generations"]))]
gen_avant = generer(modele, prompts_gen)
print(f"Avant DPO : la réponse choisie est la plus probable dans {pref_avant[0]:.0%} des paires de test "
      f"(somme des log-probabilités), {pref_avant[1]:.0%} (moyenne par token)")
print(f"Longueur moyenne des réponses générées : {np.mean([g['n_tokens'] for g in gen_avant]):.0f} tokens "
      f"({time.perf_counter() - t0:.0f} s)")

In [ ]:
# Cellule 7 : DPO avec TRL (LoRA ; la référence est le même modèle, adaptateur désactivé)
from peft import LoraConfig
from transformers import TrainerCallback
from trl import DPOConfig, DPOTrainer

class Journal(TrainerCallback):
    def __init__(self):
        self.lignes = []
    def on_log(self, args, state, control, logs=None, **kw):
        if logs:
            self.lignes.append({"pas": state.global_step, **{k: float(v) for k, v in logs.items()
                                                             if isinstance(v, (int, float))}})

args = DPOConfig(
    output_dir="sortie-dpo", seed=SEED, report_to="none",
    beta=config["beta"], max_length=config["max_length"],
    max_steps=config["n_pas"], per_device_train_batch_size=config["lot"],
    gradient_accumulation_steps=config["accumulation"], per_device_eval_batch_size=4,
    learning_rate=config["lr"], lr_scheduler_type="cosine", warmup_steps=max(1, config["n_pas"] // 20),
    fp16=FP16, gradient_checkpointing=True,
    logging_steps=max(1, config["n_pas"] // 25), eval_strategy="steps",
    eval_steps=max(1, config["n_pas"] // 5), save_strategy="no",
)
lora = LoraConfig(r=config["rang"], lora_alpha=config["alpha"], lora_dropout=0.05,
                  target_modules="all-linear", task_type="CAUSAL_LM")
journal = Journal()
entraineur = DPOTrainer(model=modele, ref_model=None, args=args, train_dataset=train, eval_dataset=test,
                        processing_class=tok, peft_config=lora, callbacks=[journal])
politique = entraineur.model
n_lora = sum(p.numel() for p in politique.parameters() if p.requires_grad)
print(f"Paramètres entraînés : {n_lora / 1e6:.1f} M ({n_lora / n_params:.1%}) ; aucune copie de référence en mémoire")

eval_avant = entraineur.evaluate()
if dispositif == "cuda":
    torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
entraineur.train()
duree = time.perf_counter() - t0
eval_apres = entraineur.evaluate()
pic = torch.cuda.max_memory_allocated() / 2**30 if dispositif == "cuda" else float("nan")
print(f"Entraînement : {duree / 60:.1f} min ({duree / config['n_pas']:.2f} s/pas), pic mémoire {pic:.2f} Gio")
for cle in ["eval_loss", "eval_rewards/chosen", "eval_rewards/rejected", "eval_rewards/margins", "eval_rewards/accuracies"]:
    print(f"{cle:26s} avant {eval_avant.get(cle, float('nan')):8.4f}   après {eval_apres.get(cle, float('nan')):8.4f}")

In [ ]:
# Cellule 8 : APRÈS, vérifier la récompense implicite à la main, puis préférences et longueurs
e = test[0]
lp_w, lp_l = logp_reponse(politique, e["prompt"], e["chosen"]), logp_reponse(politique, e["prompt"], e["rejected"])
with politique.disable_adapter():                      # π_ref : même modèle, sans le LoRA
    lp_w_ref, lp_l_ref = logp_reponse(politique, e["prompt"], e["chosen"]), logp_reponse(politique, e["prompt"], e["rejected"])
perte, r_w, r_l = perte_dpo_main(lp_w, lp_l, lp_w_ref, lp_l_ref)
print(f"Paire de test n° 0, à la main : récompense implicite choisie {r_w:+.3f}, rejetée {r_l:+.3f}, "
      f"marge {r_w - r_l:+.3f}, perte {perte:.4f}")

pref_apres = preferences_modele(politique, test)
gen_apres = generer(politique, prompts_gen)
long_avant = np.array([g["n_tokens"] for g in gen_avant]); long_apres = np.array([g["n_tokens"] for g in gen_apres])
print(f"Choisie plus probable (somme / moyenne par token) : avant {pref_avant[0]:.0%} / {pref_avant[1]:.0%}, "
      f"après {pref_apres[0]:.0%} / {pref_apres[1]:.0%}")
print(f"Longueur des réponses : avant {long_avant.mean():.0f} tokens, après {long_apres.mean():.0f} ; "
      f"réponses coupées à {config['nouveaux_tokens']} tokens : {np.mean(long_avant >= config['nouveaux_tokens']):.0%} → "
      f"{np.mean(long_apres >= config['nouveaux_tokens']):.0%}")
for i in range(min(2, len(prompts_gen))):
    print(f"\n===== {prompts_gen[i][-1]['content'][:200]}")
    print(f"--- AVANT ({gen_avant[i]['n_tokens']} tokens)\n{gen_avant[i]['texte'][:700]}")
    print(f"--- APRÈS ({gen_apres[i]['n_tokens']} tokens)\n{gen_apres[i]['texte'][:700]}")

In [ ]:
# Cellule 9 : courbes et sauvegarde
import matplotlib.pyplot as plt

tr = [l for l in journal.lignes if "loss" in l and "eval_loss" not in l]
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 4))
a1.plot([l["pas"] for l in tr], [l["loss"] for l in tr]); a1.axhline(np.log(2), ls=":", color="grey")
a1.set(xlabel="pas", ylabel="perte DPO (entraînement)", title="Perte (départ : log 2)")
for cle, nom in [("rewards/chosen", "choisies"), ("rewards/rejected", "rejetées"), ("rewards/margins", "marge")]:
    a2.plot([l["pas"] for l in tr if cle in l], [l[cle] for l in tr if cle in l], label=nom)
a2.axhline(0, color="grey", lw=0.8); a2.legend()
a2.set(xlabel="pas", ylabel="β·log π_θ/π_ref", title="Récompenses implicites (entraînement)")
a3.hist([long_avant, long_apres], bins=12, label=["avant DPO", "après DPO"])
a3.set(xlabel="tokens générés", ylabel="réponses", title="Longueur des réponses"); a3.legend()
fig.tight_layout(); fig.savefig("30b-dpo.png", dpi=120); plt.show()

with open("30b-resultats.json", "w") as f:
    json.dump(dict(config=config, eval_avant=eval_avant, eval_apres=eval_apres, duree_s=duree, pic_gio=pic,
                   preferences_avant=pref_avant, preferences_apres=pref_apres,
                   longueurs_avant=long_avant.tolist(), longueurs_apres=long_apres.tolist(),
                   journal=journal.lignes, generations_avant=gen_avant, generations_apres=gen_apres),
              f, ensure_ascii=False, indent=1)
politique.save_pretrained("adaptateur-dpo")

In [ ]:
# Cellule 10 : récupérer les résultats avant la fin de session (Colab ; sur Kaggle : onglet Output)
import shutil
shutil.make_archive("adaptateur-dpo", "zip", ".", "adaptateur-dpo")
fichiers = ["30b-resultats.json", "30b-dpo.png", "adaptateur-dpo.zip"]
try:
    from google.colab import files
    for f in fichiers:
        files.download(f)
except ImportError:
    print("Hors Colab : fichiers laissés dans le dossier courant.")